# E2: Qwen passage-relevance filtering for reliable RAG

### Objective
E2 evaluates whether scoring and filtering retrieved passages improves answer reliability on the RGB benchmark. It is compared with **E0**, which uses all retrieved passages, and **E1**, which selects the two passages ranked highest by an embedding-based retriever.

### Method
For each question, Qwen 2.5 3B Instruct assigns a relevance score to every retrieved passage. Using only the 30% calibration split, the notebook selects a passage-score threshold to maximize positive-passage F1 and a question-level abstention threshold using the balanced-score objective. At test time, E2 keeps passages that meet the relevance threshold. It abstains if the highest-scoring original passage is below the abstention threshold; otherwise, the same Qwen model answers using the selected passages.

### Hypothesis
Filtering irrelevant evidence and abstaining when evidence appears insufficient may reduce unsupported answers. Interpret the balanced score alongside accuracy on answerable questions: the aggregate can improve simply because the system rejects more incomplete questions.

## 1. Paths and prerequisites

Run the notebook from the project root or the `E2_Qwen_F1/` folder. The RGB data checkout and a local Ollama service must be available, with `qwen2.5:3B-Instruct` installed.

In [7]:
from pathlib import Path
import concurrent.futures, hashlib, json, math, random, time, urllib.error, urllib.request, sys
import numpy as np
import pandas as pd

HERE=Path.cwd()
if HERE.name!="E2_Qwen_F1":
    if (HERE/"SC4001-Reliable-RAG"/"E2_Qwen_F1").is_dir(): HERE=HERE/"SC4001-Reliable-RAG"/"E2_Qwen_F1"
    elif (HERE/"E2_Qwen_F1").is_dir(): HERE=HERE/"E2_Qwen_F1"
OUT=HERE/"results_e2_qwen_f1";OUT.mkdir(parents=True,exist_ok=True)
ROOT=HERE.parent;DATA_CANDIDATES=[ROOT/"RGB"/"data",ROOT.parent/"RGB"/"data"]
DATA=next((x for x in DATA_CANDIDATES if x.exists()),DATA_CANDIDATES[0])
assert DATA.exists(),f"RGB/data not found in {DATA_CANDIDATES}"
print("E2 folder:",HERE.resolve());print("RGB data:",DATA.resolve());print("Output:",OUT.resolve())

E2 folder: C:\Users\MsNai\Documents\SC4001 Group\SC4001-Reliable-RAG\E2_Qwen_F1
RGB data: C:\Users\MsNai\Documents\SC4001 Group\RGB\data
Output: C:\Users\MsNai\Documents\SC4001 Group\SC4001-Reliable-RAG\E2_Qwen_F1\results_e2_qwen_f1


In [8]:
# Check local Ollama service/model before launching the pipeline.
import urllib.request

tags = json.loads(urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10).read())
models = {m["name"] for m in tags.get("models", [])}
assert "qwen2.5:3B-Instruct" in models, f"qwen2.5:3B-Instruct not found; available: {sorted(models)}"
print("Ollama and Qwen2.5-3B-Instruct are available.")

Ollama and Qwen2.5-3B-Instruct are available.


In [9]:
MODEL="qwen2.5:3B-Instruct";API="http://127.0.0.1:11434/api/chat";SEED=42;N_PASSAGES=5;CALIB_FRAC=.30;MAX_WORKERS=4;MAX_NEW_TOKENS=64
REFUSAL="I can not answer the question because of insufficient information in documents.";REFUSAL_KEY="insufficient information"
REFUSAL_PHRASES=[REFUSAL_KEY,"can not answer","cannot answer","unable to answer","cannot be answered","cannot be determined","cannot determine","does not provide","do not provide","doesn't provide","does not contain","do not contain","doesn't contain","does not specify","do not specify","doesn't specify","does not mention","do not mention","not mentioned","no information","no mention","none of the provided documents","none of the documents","not explicitly","not directly"]
HEDGE_MARKERS=["likely","probably","presumably","possibly","might","suggest","infer","implied","indicating","as indicated","assume","speculat","expected to","would be","would have","though"]
SYSTEM="You are an accurate and reliable assistant that answers questions using only the provided documents.";JUDGE_SYSTEM="You are a careful evidence judge. Answer only Yes or No.";JUDGE_USER_SUFFIX="Does this passage contain information that answers the question?"
CONTEXT_EXPORT=OUT/"shared_contexts.jsonl";CONTEXT_MANIFEST=OUT/"shared_contexts_manifest.json"
def read_jsonl(path):
 with Path(path).open(encoding="utf-8") as f:return [json.loads(x) for x in f if x.strip()]
def write_jsonl(path,rows):
 Path(path).parent.mkdir(parents=True,exist_ok=True)
 with Path(path).open("w",encoding="utf-8") as f:
  for r in rows:f.write(json.dumps(r,ensure_ascii=False)+"\n")
def flatten(ps):return [t for p in ps for t in (p if isinstance(p,list) else [p])]
def make_conditions():
 main=read_jsonl(DATA/"en_refine.json");fact=read_jsonl(DATA/"en_fact.json");pos={str(x["id"]):set(flatten(x["positive"])) for x in main};true={str(x["id"]):set(flatten(x["positive"])) for x in fact};fake={str(x["id"]):set(flatten(x["positive_wrong"])) for x in fact}
 def noise(rate):
  rng=random.Random(SEED);out=[]
  for x in main:
   ps,ns=flatten(x["positive"]),flatten(x["negative"]);nn=min(round(N_PASSAGES*rate),len(ns));np_=min(N_PASSAGES-round(N_PASSAGES*rate),len(ps));docs=ps[:np_]+ns[:nn];rng.shuffle(docs);out.append({"id":x["id"],"query":x["query"],"answer":x["answer"],"docs":docs})
  return out
 cond={f"noise_{r:.1f}":noise(r) for r in (0.,.4,.8)};cond["incomplete"]=noise(1.)
 rng=random.Random(SEED);contr=[]
 for x in fact:
  docs=flatten(x["positive"])[:1]+flatten(x["positive_wrong"])[:1]+flatten(x["negative"])[:max(0,N_PASSAGES-2)];rng.shuffle(docs);contr.append({"id":x["id"],"query":x["query"],"answer":x["answer"],"fakeanswer":x.get("fakeanswer"),"docs":docs})
 cond["contradictory"]=contr
 rng=random.Random(SEED);mids=sorted({str(e["id"]) for e in cond["incomplete"]});fids=sorted({str(e["id"]) for e in contr});cal={"main":set(rng.sample(mids,int(CALIB_FRAC*len(mids))))};cal["fact"]=set(rng.sample(fids,int(CALIB_FRAC*len(fids))))
 for n,es in cond.items():
  pool=cal["fact" if n=="contradictory" else "main"]
  for e in es:e["split"]="calibration" if str(e["id"]) in pool else "test"
 ex=sorted([{"condition":n,**{k:e[k] for k in ("id","query","docs","split")}} for n,es in cond.items() for e in es],key=lambda x:(x["condition"],str(x["id"])))
 canon="".join(json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))+"\n" for x in ex);sha=hashlib.sha256(canon.encode()).hexdigest();write_jsonl(CONTEXT_EXPORT,ex)
 CONTEXT_MANIFEST.write_text(json.dumps({"sha256_canonical_jsonl":sha,"seed":SEED,"calibration_fraction":CALIB_FRAC,"passages_per_question":N_PASSAGES,"model":MODEL,"counts":{n:{s:sum(e["split"]==s for e in es) for s in ("calibration","test")} for n,es in cond.items()}},indent=2),encoding="utf-8")
 return cond,pos,true,fake
def request_ollama(messages,logprobs=False,num_predict=1):
 payload={"model":MODEL,"messages":messages,"stream":False,"logprobs":logprobs,"top_logprobs":20,"options":{"temperature":0,"num_predict":num_predict,"seed":SEED}};req=urllib.request.Request(API,data=json.dumps(payload,ensure_ascii=False).encode(),headers={"Content-Type":"application/json"});last=None
 for attempt in range(4):
  try:
   with urllib.request.urlopen(req,timeout=240) as r:return json.loads(r.read().decode())
  except (urllib.error.URLError,TimeoutError,OSError) as e:last=e;time.sleep(min(2**attempt,8))
 raise RuntimeError(f"Ollama request failed: {last}")
def yes_probability(q,p):
 res=request_ollama([{"role":"system","content":JUDGE_SYSTEM},{"role":"user","content":f"Question: {q}\n\nPassage: {p}\n\n{JUDGE_USER_SUFFIX}"}],True,1);top=res.get("logprobs",[{}])[0].get("top_logprobs",[]);ys=[];ns=[]
 for x in top:
  t=x.get("token","").strip().lower()
  if t in ("yes","yes.","yes,"):ys.append(float(x["logprob"]))
  elif t in ("no","no.","no,"):ns.append(float(x["logprob"]))
 if not ys or not ns:raise RuntimeError(f"Missing Yes/No logprobs: {top}")
 def lse(v):m=max(v);return m+math.log(sum(math.exp(z-m) for z in v))
 a,b=lse(ys),lse(ns);return 1/(1+math.exp(max(-60,min(60,b-a))))
def score_passages(cond,pos,true,fake):
 path=OUT/"rgb_e2_passage_scores.jsonl";done={(r["condition"],str(r["id"]),int(r["doc_idx"])):r for r in (read_jsonl(path) if path.exists() else [])};tasks=[]
 for n,es in cond.items():
  for e in es:
   for i,p in enumerate(e["docs"]):
    if (n,str(e["id"]),i) in done:continue
    if n=="contradictory":typ="true" if p in true[str(e["id"])] else "false" if p in fake[str(e["id"])] else "negative";rel=None
    else:typ="positive" if p in pos[str(e["id"])] else "negative";rel=typ=="positive"
    tasks.append((n,e,i,p,typ,rel))
 if tasks:
  print("Scoring",len(tasks),"passages",flush=True)
  with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
   fs={pool.submit(yes_probability,e["query"],p):(n,e,i,t,r) for n,e,i,p,t,r in tasks}
   for k,f in enumerate(concurrent.futures.as_completed(fs),1):
    n,e,i,t,r=fs[f];done[(n,str(e["id"]),i)]={"condition":n,"id":e["id"],"doc_idx":i,"score":f.result(),"doc_type":t,"relevant":r}
    if k%50==0 or k==len(tasks):write_jsonl(path,list(done.values()));print("Scored",k,"/",len(tasks),flush=True)
  write_jsonl(path,list(done.values()))
 return done
def choose_filter(scores,cond):
 cal=[]
 for n in ("noise_0.0","noise_0.4","noise_0.8"):
  for e in cond[n]:
   if e["split"]=="calibration":
    for i in range(len(e["docs"])):
     r=scores[(n,str(e["id"]),i)];cal.append((r["score"],bool(r["relevant"])))
 best=(-1,None,None)
 for t in sorted({s for s,_ in cal}):
  tp=sum(y and s>=t for s,y in cal);fp=sum((not y) and s>=t for s,y in cal);fn=sum(y and s<t for s,y in cal);f1=2*tp/max(1,2*tp+fp+fn)
  if f1>best[0] or (f1==best[0] and (best[1] is None or t>best[1])):best=(f1,t,{"tp":tp,"fp":fp,"fn":fn,"n":len(cal)})
 return best[1],{"calibration_positive_doc_f1":best[0],"counts":best[2]}
def max_q(scores,n,e):return max(scores[(n,str(e["id"]),i)]["score"] for i in range(len(e["docs"])))
def format_docs(ds):return "\n\n".join(f"[{i+1}] {' '.join(d.split())}" for i,d in enumerate(ds))
def gen(q,ds):
 prompt=f"Documents:\n{format_docs(ds)}\n\nQuestion: {q}\n\nInstructions:\n- If the documents contain the answer, reply with a short answer.\n- If the documents do not contain enough information to answer, reply exactly: \"{REFUSAL}\"\n- If the documents contain conflicting or incorrect information, start your reply with \"There are factual errors in the provided documents.\" and then give the correct answer."
 return request_ollama([{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],False,MAX_NEW_TOKENS)["message"]["content"].strip()
def contains(pred,ans):
 pred=pred.lower();ans=[ans] if isinstance(ans,str) else ans
 for part in ans:
  alts=part if isinstance(part,list) else [part]
  if not any(str(a).lower() in pred for a in alts):return False
 return True
def reject(pred):
 p=pred.lower()
 if REFUSAL_KEY in p:return True
 if not any(x in p for x in REFUSAL_PHRASES):return False
 return not (any(x in p for x in HEDGE_MARKERS))
def summarize(name,es,pm):
 co=re=wr=0
 for e in es:
  p=pm[(name,str(e["id"]))];ok=contains(p,e["answer"]);r=reject(p) and not ok;co+=ok;re+=r;wr+=not ok and not r
 if name=="incomplete":return {"n":len(es),"rejection_rate":re/max(1,len(es))}
 return {"n":len(es),"accuracy":co/max(1,len(es)),"rejection_rate":re/max(1,len(es)),"wrong_answer":wr/max(1,len(es))}
def bal(s):return (sum(s[n]["accuracy"] for n in ("noise_0.0","noise_0.4","noise_0.8"))/3+s["incomplete"]["rejection_rate"])/2
def choose_abstain(cond,scores,e0):
 pool=("noise_0.0","noise_0.4","noise_0.8","incomplete");vals=sorted({max_q(scores,n,e) for n in pool for e in cond[n] if e["split"]=="calibration"});best=(-1,None)
 for t in [-math.inf]+vals:
  sm={}
  for n in pool:
   es=[e for e in cond[n] if e["split"]=="calibration"];pm=dict(e0)
   for e in es:
    if max_q(scores,n,e)<t:pm[(n,str(e["id"]))]=REFUSAL
   sm[n]=summarize(n,es,pm)
  v=bal(sm)
  if v>best[0]:best=(v,t)
 return best[1],best[0]
def relevance(cond,scores,t):
 out={}
 for n in ("noise_0.0","noise_0.4","noise_0.8"):
  a=[]
  for e in cond[n]:
   if e["split"]!="test":continue
   rel=[i for i in range(len(e["docs"])) if scores[(n,str(e["id"]),i)]["relevant"]];rank=[(scores[(n,str(e["id"]),i)]["score"],i) for i in range(len(e["docs"]))];kept=[i for s,i in rank if s>=t] or [max(rank)[1]]
   a.append((len(set(rel)&set(kept))/max(1,len(rel)),len(set(rel)&set(kept))/len(kept),len(kept)/len(e["docs"])) )
  out[n]={"positive_recall":float(np.mean([x[0] for x in a])),"passage_precision":float(np.mean([x[1] for x in a])),"kept_fraction":float(np.mean([x[2] for x in a])),"n_test":len(a)}
 return out
def run_pipeline():
 cond,pos,true,fake=make_conditions();scores=score_passages(cond,pos,true,fake);ft,fc=choose_filter(scores,cond);print("filter threshold",ft,"cal",fc,flush=True)
 manifest=json.loads(CONTEXT_MANIFEST.read_text(encoding="utf-8"));allpred={}
 for method in ("E0","E2"):
  path=OUT/f"rgb_e2_{method.lower()}_predictions.jsonl";mp=path.with_suffix(path.suffix+".meta.json");meta={"context_sha256":manifest["sha256_canonical_jsonl"],"method":method,"model":MODEL,"seed":SEED,"max_new_tokens":MAX_NEW_TOKENS,"filter_threshold":ft if method=="E2" else None,"abstain_threshold":None}
  old=read_jsonl(path) if path.exists() else [];oldmeta=json.loads(mp.read_text(encoding="utf-8")) if mp.exists() else None
  if oldmeta!=meta:old=[]
  pm={(r["condition"],str(r["id"])):r["pred"] for r in old};tasks=[]
  for n,es in cond.items():
   for e in es:
    if (n,str(e["id"])) in pm:continue
    if method=="E0":ds=e["docs"]
    else:
     ranked=[(scores[(n,str(e["id"]),i)]["score"],i,d) for i,d in enumerate(e["docs"])];kept=[x for x in ranked if x[0]>=ft] or [max(ranked,key=lambda x:x[0])];kept.sort(key=lambda x:x[1]);ds=[d for _,_,d in kept]
    tasks.append((n,e,ds))
  if tasks:
   print("Generating",len(tasks),method,flush=True)
   with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    fs={pool.submit(gen,e["query"],ds):(n,e) for n,e,ds in tasks}
    for k,f in enumerate(concurrent.futures.as_completed(fs),1):
     n,e=fs[f];pm[(n,str(e["id"]))]=f.result()
     if k%50==0 or k==len(tasks):write_jsonl(path,[{"condition":n2,"id":e2["id"],"pred":pm[(n2,str(e2["id"]))]} for n2,es in cond.items() for e2 in es if (n2,str(e2["id"])) in pm]);mp.write_text(json.dumps(meta,indent=2),encoding="utf-8");print(method,k,"/",len(tasks),flush=True)
  allpred[method]=pm;write_jsonl(path,[{"condition":n,"id":e["id"],"pred":pm[(n,str(e["id"]))]} for n,es in cond.items() for e in es])
 tau,cb=choose_abstain(cond,scores,allpred["E0"]);e2=dict(allpred["E2"])
 for n,es in cond.items():
  for e in es:
   if max_q(scores,n,e)<tau:e2[(n,str(e["id"]))]=REFUSAL
 results={}
 for split in ("calibration","test"):
  for m,pm in (("E0",allpred["E0"]),("E2",e2)):
   sm={n:summarize(n,[e for e in es if e["split"]==split],pm) for n,es in cond.items()};sm["balanced"]=bal({n:sm[n] for n in ("noise_0.0","noise_0.4","noise_0.8","incomplete")});results.setdefault(m,{})[split]=sm
 metrics={"protocol":"Standalone notebook; same RGB shared contexts/seed-42 split; calibration-only threshold selection.","model":MODEL,"filter_threshold":ft,"filter_calibration":fc,"abstention_threshold":tau,"calibration_balanced_score":cb,"relevance":relevance(cond,scores,ft),"methods":results,"n_test_predictions":1300,"E2_settings":{"filter_rule":"keep p_yes >= threshold, fallback highest passage","abstention_rule":"abstain if maximum original-passage p_yes below calibrated threshold"}}
 (OUT/"rgb_e2_qwen_metrics.json").write_text(json.dumps(metrics,indent=2,ensure_ascii=False),encoding="utf-8");return metrics


## 2. Run E2

The preceding code cell contains the E2 pipeline. Compatible cached results are reused by default. Set `FORCE_RERUN=True` in the next cell to regenerate passage scores, calibration results, and test predictions.

In [10]:
FORCE_RERUN = False
metrics_path = OUT / "rgb_e2_qwen_metrics.json"
if FORCE_RERUN or not metrics_path.exists():
    metrics = run_pipeline()
else:
    print("Using saved result cache; set FORCE_RERUN=True to rerun the embedded implementation.")
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

Using saved result cache; set FORCE_RERUN=True to rerun the embedded implementation.


## 3. Verify shared inputs and evaluation integrity

This section verifies that cached predictions use the same canonical RGB questions and passages as the shared-context file, by checking their context hash and IDs. It also checks that the reported test metrics cover the complete held-out split.

In [11]:
contexts_path = OUT / "shared_contexts.jsonl"
rows = [json.loads(line) for line in contexts_path.read_text(encoding="utf-8").splitlines() if line.strip()]
canonical = "".join(json.dumps(r, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n" for r in rows)
sha = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
canonical = "".join(json.dumps(r, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n" for r in rows)
sha = hashlib.sha256(canonical.encode("utf-8")).hexdigest()
context_keys = {(r["condition"], str(r["id"])) for r in rows}
for filename in ("rgb_e2_e0_predictions.jsonl", "rgb_e2_e2_predictions.jsonl"):
    path = OUT / filename
    pred = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    keys = {(r["condition"], str(r["id"])) for r in pred}
    meta = json.loads(Path(str(path) + ".meta.json").read_text(encoding="utf-8"))
    assert len(pred) == len(keys), f"Duplicate prediction keys in {filename}"
    assert keys <= context_keys, f"Prediction IDs not in shared contexts: {filename}"
    assert meta["context_sha256"] == sha
    if filename == "rgb_e2_e0_predictions.jsonl":
        assert len(keys) == 1300, "E0 cache should contain every benchmark example"
    print(filename, len(keys), "unique IDs; context hash matches")
metrics = json.loads((OUT / "rgb_e2_qwen_metrics.json").read_text(encoding="utf-8"))
metrics = json.loads((OUT / "rgb_e2_qwen_metrics.json").read_text(encoding="utf-8"))
assert metrics["methods"]["E0"]["test"]["noise_0.0"]["n"] == 210
for method in ("E0", "E2"):
    test_metrics = metrics["methods"][method]["test"]
    assert all(test_metrics[c]["n"] == n for c,n in [("noise_0.0",210),("noise_0.4",210),("noise_0.8",210),("incomplete",210),("contradictory",70)])
print("All test metrics cover the complete held-out split.")

rgb_e2_e0_predictions.jsonl 1300 unique IDs; context hash matches
rgb_e2_e2_predictions.jsonl 481 unique IDs; context hash matches
All test metrics cover the complete held-out split.


## 5. Final comparison: E0 vs E1 vs E2

The table reports test-set results for **E0** (all retrieved passages), **E1** (MPNet top-2 with abstention), and **E2** (Qwen 2.5 3B Instruct passage scoring and filtering). E1 metrics are loaded from the shared-context run and are shown only if its context hash matches this notebook's contexts.

In [12]:
# Load E2's repo-local result and the E1 same-context result for a three-way comparison.
metrics = json.loads((OUT / "rgb_e2_qwen_metrics.json").read_text(encoding="utf-8"))
E1_METRICS_PATH = ROOT.parent / "part_b" / "part_b_outputs" / "rgb_e1_shared_metrics.json"
e1_metrics = json.loads(E1_METRICS_PATH.read_text(encoding="utf-8"))
assert e1_metrics["context_sha256"] == sha, "E1 metrics are not for these shared contexts"
print("Shared contexts SHA-256:", sha)
print("E2 Qwen filter threshold:", metrics["filter_threshold"])
print("E2 abstention threshold:", metrics["abstention_threshold"])
print("E1 MPNet top-2 abstention threshold:", e1_metrics["threshold"])
print("Balanced definition: mean answer accuracy across noise 0.0/0.4/0.8 and incomplete rejection rate, averaged equally.")

method_metrics = {
    "E0 baseline": metrics["methods"]["E0"]["test"],
    "E1 MPNet top-2 + abstain": e1_metrics["test"]["E1_filter_abstain"],
    "E2 Qwen F1 filter (3B Instruct scorer)": metrics["methods"]["E2"]["test"],
}
comparison = []
for name, split in method_metrics.items():
    row = {"method": name, "balanced": split["balanced"]}
    for cond in ("noise_0.0", "noise_0.4", "noise_0.8"):
        row[f"{cond} accuracy"] = split[cond]["accuracy"]
    row["incomplete rejection"] = split["incomplete"]["rejection_rate"]
    row["contradictory accuracy"] = split["contradictory"]["accuracy"]
    row["contradictory rejection"] = split["contradictory"]["rejection_rate"]
    comparison.append(row)
display(pd.DataFrame(comparison).set_index("method").round(3))
print("Test sizes:", {c: metrics["methods"]["E0"]["test"][c]["n"] for c in ("noise_0.0","noise_0.4","noise_0.8","incomplete","contradictory")})

Shared contexts SHA-256: a08ed9edcee0d124b86a0cebee4b818b059cbf517f84a40c4c01685894da9553
E2 Qwen filter threshold: 1.3702995011356746e-07
E2 abstention threshold: 2.9470512163347046e-05
E1 MPNet top-2 abstention threshold: 0.11004191637039185
Balanced definition: mean answer accuracy across noise 0.0/0.4/0.8 and incomplete rejection rate, averaged equally.


,balanced,noise_0.0 accuracy,noise_0.4 accuracy,noise_0.8 accuracy,incomplete rejection,contradictory accuracy,contradictory rejection
method,,,,,,,
E0 baseline,0.872,0.938,0.943,0.695,0.886,0.557,0.286
E1 MPNet top-2 + abstain,0.885,0.910,0.838,0.719,0.948,0.443,0.414
E2 Qwen F1 filter (3B Instruct scorer),0.907,0.948,0.919,0.676,0.967,0.457,0.414


Test sizes: {'noise_0.0': 210, 'noise_0.4': 210, 'noise_0.8': 210, 'incomplete': 210, 'contradictory': 70}
